In [1]:
!nvidia-smi

Sun Jun 28 15:49:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.5 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from datasets import load_dataset
dataset = load_dataset(
    "json",
    data_files = {
        "train": "/content/drive/MyDrive/TelecomFaultLorafinetuning/data/processed/train.jsonl",
        "test": "/content/drive/MyDrive/TelecomFaultLorafinetuning/data/processed/test.jsonl"
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 3200
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 400
    })
})

In [6]:
dataset["train"][0]

{'instruction': 'Classify the telecom fault,identify the root cause,determine severity,and recommend resolution steps.',
 'input': 'Network Element: AMF_5336\nRegion: Hyderabad-HITEC-City\nAlarm Type: Process Crash\nKPI Summary: packet_loss=7.89%, latency=234.76ms, call_drop_rate=2.38%, throughput=51.48Mbps, rrc_failure_rate=4.5%, handover_failure_rate=3.32%, prb_utilization=80.37%, sinr=18.86dB\nRaw Log: - 1122144022 2005.07.23 R22-M0-N3-C:J02-U01 2005-07-23-11.40.22.071233 R22-M0-N3-C:J02-U01 RAS KERNEL INFO generating core.31691',
 'output': 'Fault Category: Application Crash\nRoot Cause: Runtime process failure\nSeverity: High\nResolution Steps: Inspect core dump logs, restart affected process, and validate service health'}

In [7]:
def format_prompt(example):
    text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

    return {"text": text}


dataset = dataset.map(format_prompt)

Map:   0%|          | 0/3200 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [8]:
print(dataset["train"][0]["text"])

### Instruction:
Classify the telecom fault,identify the root cause,determine severity,and recommend resolution steps.

### Input:
Network Element: AMF_5336
Region: Hyderabad-HITEC-City
Alarm Type: Process Crash
KPI Summary: packet_loss=7.89%, latency=234.76ms, call_drop_rate=2.38%, throughput=51.48Mbps, rrc_failure_rate=4.5%, handover_failure_rate=3.32%, prb_utilization=80.37%, sinr=18.86dB
Raw Log: - 1122144022 2005.07.23 R22-M0-N3-C:J02-U01 2005-07-23-11.40.22.071233 R22-M0-N3-C:J02-U01 RAS KERNEL INFO generating core.31691

### Response:
Fault Category: Application Crash
Root Cause: Runtime process failure
Severity: High
Resolution Steps: Inspect core dump logs, restart affected process, and validate service health


In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [11]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/models/tinyllama-telecom-lora",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=25,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    fp16=False, # Changed to False
    bf16=True,  # Changed to True
    max_length=512,      # changed here
    packing=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2472 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.482732,0.475198,0.477196,490419.000000,0.819013
400,0.458691,0.455567,0.457525,980003.000000,0.823699
600,0.449493,0.444251,0.446902,1471315.000000,0.826594
800,0.436527,0.439033,0.441032,1960006.000000,0.827560


TrainOutput(global_step=800, training_loss=0.4904345989227295, metrics={'train_runtime': 9100.469, 'train_samples_per_second': 0.703, 'train_steps_per_second': 0.088, 'total_flos': 1.2314263729250304e+16, 'train_loss': 0.4904345989227295, 'epoch': 2.0})

In [12]:
trainer.model.save_pretrained("/content/drive/MyDrive/TelecomFaultLorafinetuning/models/tinyllama-telecom-lora-adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/TelecomFaultLorafinetuning/models/tinyllama-telecom-lora-adapter")

('/content/drive/MyDrive/TelecomFaultLorafinetuning/models/tinyllama-telecom-lora-adapter/tokenizer_config.json',
 '/content/drive/MyDrive/TelecomFaultLorafinetuning/models/tinyllama-telecom-lora-adapter/chat_template.jinja',
 '/content/drive/MyDrive/TelecomFaultLorafinetuning/models/tinyllama-telecom-lora-adapter/tokenizer.json')

In [14]:
test_prompt = """### Instruction:
Classify the telecom fault, identify the root cause, determine severity, and recommend resolution steps.

### Input:
Network Element: eNodeB_2041
Region: Gurugram-Sector-44
Alarm Type: Backhaul Packet Loss Alarm
KPI Summary: packet_loss=16.5%, latency=245ms, call_drop_rate=6.2%, throughput=8Mbps
Raw Log: PacketResponder terminated unexpectedly due to transport stream failure

### Response:
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=180,
    temperature=0.2,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=180) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


### Instruction:
Classify the telecom fault, identify the root cause, determine severity, and recommend resolution steps.

### Input:
Network Element: eNodeB_2041
Region: Gurugram-Sector-44
Alarm Type: Backhaul Packet Loss Alarm
KPI Summary: packet_loss=16.5%, latency=245ms, call_drop_rate=6.2%, throughput=8Mbps
Raw Log: PacketResponder terminated unexpectedly due to transport stream failure

### Response:
Fault Category: Backhaul Issue
Root Cause: Packet forwarding interruption on transport path
Severity: Critical
Resolution Steps: Escalate to transport team for backhaul path validation
